# boring — CUDA quantized prefill benchmark (fp32 vs half WMMA)

Times `examples/cuda_prefill_bench/` (20 back-to-back quantized `kernel.tensors.linear` prefill calls, one synchronization) with the exact fp32 tiled kernel and with the opt-in half WMMA kernel (`[tensor.linear.prefill] precision = "half"`).

**Runtime**: any NVIDIA GPU with sm_70+ (T4 works). Read ms/call and TFLOP/s; `y[0]`/`y[last]` should agree between the two runs to about 1e-3 relative.

Note: the fp32 baseline is a simple 16x16 shared-memory tile, not cuBLAS; the half kernel is untuned (64x64 tiles, float staging), so treat the ratio as a first measurement, not a final one.

In [ ]:
# Allocated GPU + CUDA driver version
!nvidia-smi

In [ ]:
# nvcc is not always on PATH on Colab: look in /usr/local/cuda*, install the toolkit as a last resort
import glob, os, shutil, subprocess
if not shutil.which("nvcc"):
    found = sorted(glob.glob("/usr/local/cuda*/bin/nvcc"))
    if found:
        os.environ["PATH"] = os.path.dirname(found[-1]) + ":" + os.environ["PATH"]
if not shutil.which("nvcc"):
    print("nvcc not found: installing the CUDA toolkit (a few minutes)...")
    subprocess.run("apt-get update -qq && apt-get install -y -qq nvidia-cuda-toolkit", shell=True, check=True)
assert shutil.which("nvcc"), "nvcc still missing: check that the runtime has a GPU (Runtime > Change runtime type)"
print(shutil.which("nvcc"))
!nvcc --version


In [ ]:
# Rust is NOT preinstalled on Colab, unlike CUDA
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version && cargo --version

## Clone the repo

The project is cloned from GitHub (`github.com/mlanoe/boring`). Change `BRANCH` to the branch you want to test (push it first).

In [ ]:
BRANCH = "main"  # <- replace with your working branch
!git clone --branch $BRANCH --depth 1 https://github.com/mlanoe/boring.git
%cd boring

In [ ]:
# Build the boring compiler itself (a Rust CLI, nothing GPU-specific here)
!cargo build --release --quiet
BORING = "./target/release/boring"
!$BORING --help | head -5

In [ ]:
import pathlib, re, subprocess
PROJECT = pathlib.Path('examples/cuda_prefill_bench')
SOURCE = PROJECT / 'main.br'
out = subprocess.run(['nvcc','--version'],capture_output=True,text=True).stdout
m = re.search(r'release (\d+)\.(\d+)', out)
FEATURE = f'cuda-{m.group(1)}{int(m.group(2)):02d}0'
print('cudarc feature:', FEATURE)

def build(toml):
    (PROJECT / 'boring.toml').write_text(toml)
    subprocess.run([BORING, 'build', '--target', 'cuda', str(SOURCE)], check=True)
    d = PROJECT / 'main_cuda'
    ct = d / 'Cargo.toml'
    ct.write_text(re.sub(r'(cudarc\s*=\s*\{[^}]*features\s*=\s*\[)([^\]]*)(\])', lambda mo: f'{mo.group(1)}"driver", "nvrtc", "cublas", "{FEATURE}"{mo.group(3)}', ct.read_text()))
    subprocess.run('cargo build --release --quiet 2>&1 | tail -3', shell=True, cwd=d, check=True)
    return d

def bench(label, toml, shapes, formats):
    d = build(toml)
    for fmt in formats:
        for shape in shapes:
            r = subprocess.run(f'cargo run --release --quiet -- {shape} {fmt} 2>/dev/null', shell=True, cwd=d, capture_output=True, text=True)
            print(f'[{label:4}]', r.stdout.strip() or r.stderr[-300:])

SHAPES = ['512x896x4864', '512x4864x896', '512x2048x11008']
FORMATS = ['q8_0', 'q4_k', 'q6_k']
HEADER = '[package]\nname = "cuda_prefill_bench"\n'


## Exact fp32 baseline

In [ ]:
bench('fp32', HEADER, SHAPES, FORMATS)

## Half WMMA (opt-in)

In [ ]:
bench('half', HEADER + '\n[tensor.linear.prefill]\nprecision = "half"\n', SHAPES, FORMATS)